# 07 — OE Image Inversion with Dask

This notebook demonstrates OE-based image inversion using `dask_oe_engine`.  
It uses the same Sentinel-2 Wadden Sea dataset as NB04, but replaces the
lmfit pixel-by-pixel loop with JAX-vmapped Gauss-Newton OE — providing
not just retrieved parameters but also **uncertainty maps** (posterior σ),
**averaging kernel maps** (how data-driven each pixel is), and a
**chi-squared map** (goodness of fit).

Key differences vs NB04:
- Forward model: `albert_mobley_jax` (JAX, exact Jacobians) instead of `albert_mobley` (lmfit)
- Retrieved params: C_0, C_Y, C_Mie, zB (vs. zB only in NB04)
- Bottom: mean of low-tide image as fixed representative spectrum (vs. per-pixel in NB04)
- Outputs: xarray Dataset with x_hat, σ, A_diag, chi² per pixel

In [ ]:
import os
import numpy as np
import xarray as xr
import lmfit
import matplotlib.pyplot as plt

from bio_optics.inversion import oe_engine, dask_oe_engine
from bio_optics.reflectance import albert_mobley_jax

## Get data

Same Sentinel-2 Wadden Sea scene as NB04: two time steps at the same location.
Time 0 = low tide (benthic reflectance visible), Time 1 = high tide (water-leaving Rrs).

In [ ]:
dataset = xr.open_dataset(os.path.join(os.getcwd(), 'example_data/S2L2A_example.nc'))
dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dataset[['B04', 'B03', 'B02']].isel(time=0).to_array().plot.imshow(robust=True, ax=axes[0])
axes[0].set_title('Low tide (benthic albedo visible)')
dataset[['B04', 'B03', 'B02']].isel(time=1).to_array().plot.imshow(robust=True, ax=axes[1])
axes[1].set_title('High tide (water-leaving Rrs — what we invert)')
plt.tight_layout()
plt.show()

## Prepare arrays

- `Rrs_image`: high-tide reflectance divided by π → Rrs [sr⁻¹], shape (n_rows, n_cols, n_obs)
- `bottom_albedo`: mean of low-tide image (raw, **not** divided by π → albedo [dimensionless])
  used as a representative benthic spectrum inserted into the precomputed lookup table.

In [ ]:
wavelengths = np.array([490, 560, 665, 705, 740, 783, 842, 865])

band_vars = list(dataset.data_vars)[1:]   # skip the first variable (time or similar)

# High-tide: divide by π to get Rrs [sr-1]
Rrs_image = np.stack(
    [dataset.isel(time=1)[v].values for v in band_vars], axis=-1
) / np.pi
print('Rrs_image shape:', Rrs_image.shape)   # (n_rows, n_cols, n_obs)

# Low-tide: raw reflectance factor (albedo, NOT divided by π)
albedo_image = np.stack(
    [dataset.isel(time=0)[v].values for v in band_vars], axis=-1
)
bottom_albedo = np.nanmean(albedo_image.reshape(-1, len(wavelengths)), axis=0)
print('bottom_albedo:', np.round(bottom_albedo, 4))

## Precompute

Run the spectral lookup table resampling once, then replace the first bottom-type
column in `R_b_i` with the measured mean low-tide albedo — mirroring NB04's approach
of using the benthic image as bottom input.

In [ ]:
import jax.numpy as jnp

pre = albert_mobley_jax.precompute(wavelengths)

# Replace first bottom-type spectrum with the measured mean benthic albedo
R_b_i_updated = np.array(pre['R_b_i'])          # (n_wavelengths, 6)
R_b_i_updated[:, 0] = bottom_albedo
pre['R_b_i'] = jnp.array(R_b_i_updated)

print('R_b_i[:, 0] (our benthic spectrum):', np.round(np.array(pre['R_b_i'][:, 0]), 4))

## Set up parameters

All parameters needed by `albert_mobley_jax`. We retrieve **C_0, C_Y, C_Mie, zB**
with log-transforms (lognormal prior); everything else is fixed.

Bottom configuration: `f_0=1` (100% first bottom type), `B_0=1/π` (Lambertian),
so `Rrs_b = bottom_albedo / π` — same convention as NB04.

In [ ]:
params = lmfit.Parameters()

# --- Geometry ---
params.add('theta_sun',  value=np.radians(30), vary=False)
params.add('theta_view', value=np.radians(0),  vary=False)
params.add('n1',         value=1.0,            vary=False)
params.add('n2',         value=1.33,           vary=False)
params.add('kappa_0',    value=1.0546,         vary=False)

# --- Water constituents (retrieved) ---
params.add('C_0',   value=0.5,  vary=True)   # phytoplankton [mg/m3]
params.add('C_Y',   value=0.1,  vary=True)   # CDOM absorption at 440 nm [m-1]
params.add('C_Mie', value=0.1,  vary=True)   # Mie particles [g/m3]

# --- Water constituents (fixed) ---
params.add('C_1',   value=0.0,  vary=False)
params.add('C_2',   value=0.0,  vary=False)
params.add('C_3',   value=0.0,  vary=False)
params.add('C_4',   value=0.0,  vary=False)
params.add('C_5',   value=0.0,  vary=False)
params.add('C_X',   value=0.0,  vary=False)  # type-I (spectrally flat) particles

# --- IOP parameters (fixed) ---
params.add('S',                  value=0.014,       vary=False)  # CDOM spectral slope [nm-1]
params.add('S_NAP',              value=0.011,       vary=False)  # NAP spectral slope [nm-1]
params.add('lambda_0',           value=440.0,       vary=False)  # CDOM reference wavelength [nm]
params.add('K',                  value=0.0,         vary=False)  # temperature correction
params.add('T_W',                value=18.0,        vary=False)  # water temperature [°C]
params.add('T_W_0',              value=20.0,        vary=False)  # reference temperature [°C]
params.add('a_NAP_spec_lambda_0', value=0.041,      vary=False)  # NAP specific absorption at lambda_0
params.add('bb_phy_spec',        value=0.0010,      vary=False)  # phytoplankton backscattering [m2/mg]
params.add('bb_Mie_spec',        value=0.0042,      vary=False)  # Mie particle backscattering [m2/g]
params.add('bb_X_spec',          value=0.0086,      vary=False)  # type-I particle backscattering
params.add('lambda_S',           value=500.0,       vary=False)  # Mie reference wavelength [nm]
params.add('n',                  value=-1.0,        vary=False)  # Angström exponent

# --- Bottom (fixed: 100% first type = measured benthic albedo) ---
params.add('f_0', value=1.0,      vary=False)
params.add('f_1', value=0.0,      vary=False)
params.add('f_2', value=0.0,      vary=False)
params.add('f_3', value=0.0,      vary=False)
params.add('f_4', value=0.0,      vary=False)
params.add('f_5', value=0.0,      vary=False)
params.add('B_0', value=1/np.pi,  vary=False)
params.add('B_1', value=1/np.pi,  vary=False)
params.add('B_2', value=1/np.pi,  vary=False)
params.add('B_3', value=1/np.pi,  vary=False)
params.add('B_4', value=1/np.pi,  vary=False)
params.add('B_5', value=1/np.pi,  vary=False)

# --- Depth (retrieved) ---
params.add('zB', value=2.0, vary=True)   # water depth [m]

print('Free parameters:', [n for n in params if params[n].vary])

## Build InversionSetup

- `log_params`: C_0, C_Y, C_Mie, zB are retrieved in log-space (lognormal prior)
  → prevents negative values during iteration; concentrations and depth can't go negative.
- `sigma_a`: in **retrieval space** (always). For log-params this means relative
  (fractional) uncertainty: 1.0 = ±100% (factor-of-e), 0.69 ≈ factor-of-two.
- `noise`: measurement uncertainty [sr⁻¹]. We use 0.001 sr⁻¹ as a conservative
  estimate for Sentinel-2 L2A over water (dominated by AtCorr residuals).

In [ ]:
log_params = ['C_0', 'C_Y', 'C_Mie', 'zB']

# sigma_a in retrieval space:
# log-params → relative (fractional) uncertainty
# linear params → physical std (none here)
sigma_a = {
    'C_0':   1.0,   # ±100% relative uncertainty (very loose for unknown chl)
    'C_Y':   1.0,
    'C_Mie': 1.0,
    'zB':    0.7,   # ≈ factor-of-two depth uncertainty
}

noise = 0.001   # measurement std [sr-1]

# Build projected forward function + prior arrays (once, on main process)
all_names = list(params.keys())
f_vec = albert_mobley_jax.make_forward_vec(all_names, pre)

setup = oe_engine.build_inversion(
    params, f_vec, sigma_a,
    log_params=log_params,
)

print('fit_names:', setup.fit_names)
print('x_a (retrieval space):', np.round(np.array(setup.x_a), 3))
print('log_mask:             ', np.array(setup.log_mask))

## Run image inversion

`invert_image` splits the image into tiles of `tile_size` pixels, dispatches each
tile as a Dask task, and within each task runs `jax.vmap(solve)` over all pixels
simultaneously.  The first tile will trigger JAX JIT compilation (~seconds); all
subsequent tiles reuse the compiled XLA program.

The default `scheduler='synchronous'` is single-threaded — JAX parallelises
internally via XLA. Switch to `'threads'` to overlap tile I/O with computation.

In [ ]:
results = dask_oe_engine.invert_image(
    Rrs_image,
    setup,
    noise=noise,
    n_iter=15,
    lm_damping=0.0,      # not needed with log-transforms
    tile_size=4096,      # pixels per Dask task (small for quick demo)
    store_y_hat=False,
    scheduler='synchronous',
)

print('Keys:', list(results.keys()))
print('x_hat shape:', results['x_hat'].shape)    # (n_rows, n_cols, n_fit)
print('chi2  shape:', results['chi2'].shape)      # (n_rows, n_cols)

## Convert to xarray Dataset

In [ ]:
ds = dask_oe_engine.to_dataset(results)
ds

## Goodness of fit — χ²

χ² ≈ 1 means the residuals match the assumed noise level.  
χ² >> 1 flags model–data mismatch: cloud shadows, emergent vegetation, optically
deep pixels where the shallow-water model is inappropriate, etc.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ds['chi2'].plot(ax=ax, vmin=0, vmax=5, cmap='RdYlGn_r')
ax.set_title('χ² (goodness of fit — ideal ≈ 1)')
plt.tight_layout()
plt.show()

print(f'Median χ²: {float(np.nanmedian(results["chi2"])):.2f}')
print(f'Pixels with χ² > 3: {(results["chi2"] > 3).sum()} / {results["chi2"].size}')

## Retrieved parameters

In [ ]:
param_labels = {
    'C_0':   ('Chlorophyll C₀', 'mg m⁻³', 'Greens'),
    'C_Y':   ('CDOM C_Y',       'm⁻¹',    'YlOrBr'),
    'C_Mie': ('SPM C_Mie',      'g m⁻³',  'Oranges'),
    'zB':    ('Depth z_B',      'm',       'Blues_r'),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (param, (label, unit, cmap)) in zip(axes.flat, param_labels.items()):
    da = ds['x_hat'].sel(param=param)
    da.plot(ax=ax, cmap=cmap, robust=True)
    ax.set_title(f'{label} [{unit}]')

plt.suptitle('Retrieved parameters (physical space)', y=1.01)
plt.tight_layout()
plt.show()

## Posterior uncertainty — σ

σ is the posterior standard deviation in **physical space** (after delta-method
conversion for log-params).  Compare σ to the prior σ_a to see how much the
data constrained each parameter.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (param, (label, unit, cmap)) in zip(axes.flat, param_labels.items()):
    da = ds['sigma'].sel(param=param)
    da.plot(ax=ax, cmap='Purples', robust=True)
    ax.set_title(f'σ({label}) [{unit}]')

plt.suptitle('Posterior uncertainty σ (physical space)', y=1.01)
plt.tight_layout()
plt.show()

## Averaging kernel diagonal — A

A[i,i] ∈ [0, 1] — fraction of the retrieved value that comes from the data
(vs. the prior).  A ≈ 1 means the parameter is fully data-driven; A ≈ 0 means
the retrieval mostly reflects the prior assumption.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (param, (label, unit, _)) in zip(axes.flat, param_labels.items()):
    da = ds['A_diag'].sel(param=param)
    da.plot(ax=ax, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(f'A[{param},{param}] (data fraction)')

plt.suptitle('Averaging kernel diagonal (0 = prior, 1 = data)', y=1.01)
plt.tight_layout()
plt.show()

# DFS = sum of A diagonal = total degrees of freedom for signal
A_diag = results['A_diag']   # (n_rows, n_cols, n_fit)
dfs_image = A_diag.sum(axis=-1)
print(f'Median DFS: {np.nanmedian(dfs_image):.2f} / {len(setup.fit_names)} parameters')

## Per-parameter summary statistics

In [ ]:
print(f'{'Parameter':10s}  {'median x_hat':>12s}  {'median σ':>10s}  {'median A[i,i]':>13s}')
print('-' * 52)
for i, name in enumerate(setup.fit_names):
    x_med  = np.nanmedian(results['x_hat'][..., i])
    s_med  = np.nanmedian(results['sigma'][..., i])
    a_med  = np.nanmedian(results['A_diag'][..., i])
    print(f'{name:10s}  {x_med:12.4f}  {s_med:10.4f}  {a_med:13.3f}')